<a href="https://colab.research.google.com/github/mmomayck/GeoAI/blob/main/Hackathon_burned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Download the dataset (image and building footprints) from Zenodo
!mkdir Dataset
!wget  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1  -O Dataset/Image.tif
!wget  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1 -O Dataset/GT.gpkg

# Install necessary packages
!pip install rasterio
!pip install dbfread

--2026-06-10 05:49:47--  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.103.118, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6765878 (6.5M) [image/tiff]
Saving to: ‘Dataset/Image.tif’

Dataset/Image.tif   100%[===================>]   6.45M   674KB/s    in 11s     

2026-06-10 05:49:58 (608 KB/s) - ‘Dataset/Image.tif’ saved [6765878/6765878]

--2026-06-10 05:49:59--  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.103.118, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 958464 (936K) [application/octet-stream]
Saving to: ‘Dataset/GT.gpkg’

Dataset/GT.gpkg     100%[===================>] 936.00K   561KB/s    in 1.7s    

2

In [3]:
# Import the libraries
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
from tqdm.notebook import tqdm
import folium
from folium import plugins

In [4]:
import os
import glob

folder_path = '/content/drive/MyDrive/Burned_S2'

file_list = glob.glob(os.path.join(folder_path, '*'))

print(f"พบไฟล์ทั้งหมด: {len(file_list)} ไฟล์\n")


for file_path in file_list:

    file_name = os.path.basename(file_path)
    print(file_name)
import re

def extract_date_from_filename(filename):
    match = re.search(r'S\d{1}[A-Z]_(\d{8})', filename)
    if match:
        return match.group(1)
    return None

# Create a list of (date, filename) tuples
files_with_dates = []
for file_path in file_list:
    file_name = os.path.basename(file_path)
    date_str = extract_date_from_filename(file_name)
    if date_str:
        files_with_dates.append((date_str, file_name))

# Sort the list by date
sorted_files_with_dates = sorted(files_with_dates, key=lambda x: x[0])

print("ไฟล์ที่เรียงตามวันที่:")
for date_str, file_name in sorted_files_with_dates:
    print(f"{date_str}: {file_name}")

พบไฟล์ทั้งหมด: 23 ไฟล์

Burned_S2_20260101_20260131_thailand.sbx
Burned_S2_20260101_20260131_thailand.sbn
Burned_S2_20260101_20260131_thailand.cpg
Burned_S2_20260101_20260131_thailand.shx
Burned_S2_20260101_20260131_thailand.prj
Burned_S2_20260101_20260131_thailand.dbf
Burned_S2_20260101_20260131_thailand.shp.xml
Burned_S2_20260101_20260131_thailand.shp
Burned_S2_20260201_20260228_thailand.prj
Burned_S2_20260201_20260228_thailand.cpg
Burned_S2_20260201_20260228_thailand.shx
Burned_S2_20260201_20260228_thailand.dbf
Burned_S2_20260201_20260228_thailand.shp.xml
Burned_S2_20260201_20260228_thailand.sbx
Burned_S2_20260201_20260228_thailand.sbn
Burned_S2_20260301_20260331_thailand.prj
Burned_S2_20260301_20260331_thailand.sbx
Burned_S2_20260301_20260331_thailand.shp.xml
Burned_S2_20260301_20260331_thailand.cpg
Burned_S2_20260401_20260430_thailand.prj
Burned_S2_20260401_20260430_thailand.cpg
Burned_S2_20260401_20260430_thailand.shp.xml
Burned_S2_20260401_20260430_thailand.sbx
ไฟล์ที่เรียงตามวั

In [5]:
import geopandas as gpd

shapefile_path = '/content/drive/MyDrive/Burned_S2/Burned_S2_20260101_20260131_thailand.shp'

gdf = gpd.read_file(shapefile_path)

gdf.head()

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


,BURN,TB_IDN,TB_TN,AP_TN,PV_TN,PV_EN,RE_NESDB,RE_ROYIN,Area_rai,RP_CODE,RP_NAME,ClusterFin,name,LU_CODE,LU_Name,Area_Map,Shape_Leng,Shape_Area,Source,geometry
0,1,710705,ต.ชะแล,อ.ทองผาภูมิ,จ.กาญจนบุรี,Kanchanaburi,West,West,4.234471,1,ป่าอนุรักษ์,12,กลุ่มป่ารอบเขื่อนศรีนครินทร์,O000,อื่น ๆ,4,0.005463,5.689347e-07,GISTDA,"MULTIPOLYGON Z (((98.8233 14.84959 0, 98.82324..."
1,1,710706,ต.ห้วยเขย่ง,อ.ทองผาภูมิ,จ.กาญจนบุรี,Kanchanaburi,West,West,2.752067,1,ป่าอนุรักษ์,None,None,O000,อื่น ๆ,3,0.003636,3.695032e-07,GISTDA,"POLYGON Z ((98.59019 14.69326 0, 98.59019 14.6..."
2,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.052637,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.000481,7.043484e-09,GISTDA,"POLYGON Z ((99.69227 13.91523 0, 99.69227 13.9..."
3,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.406916,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.002872,5.444515e-08,GISTDA,"MULTIPOLYGON Z (((99.68428 13.8941 0, 99.68427..."
4,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.026425,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.000362,3.535867e-09,GISTDA,"POLYGON Z ((99.68317 13.90802 0, 99.68301 13.9..."


In [6]:
gdf.geometry

,geometry
0,"MULTIPOLYGON Z (((98.8233 14.84959 0, 98.82324..."
1,"POLYGON Z ((98.59019 14.69326 0, 98.59019 14.6..."
2,"POLYGON Z ((99.69227 13.91523 0, 99.69227 13.9..."
3,"MULTIPOLYGON Z (((99.68428 13.8941 0, 99.68427..."
4,"POLYGON Z ((99.68317 13.90802 0, 99.68301 13.9..."
...,...
47563,"MULTIPOLYGON Z (((101.80446 15.87247 0, 101.80..."
47564,"POLYGON Z ((101.80442 15.87247 0, 101.80449 15..."
47565,"POLYGON Z ((101.80442 15.87247 0, 101.80449 15..."
47566,"POLYGON Z ((101.88476 15.89345 0, 101.88476 15..."


In [7]:
print(f"ระบบพิกัด burned: {gdf.crs}")

ระบบพิกัด burned: EPSG:4326


In [ ]:
# 1. กำหนดขนาดแผนที่
fig, ax = plt.subplots(figsize=(8, 10))

# 2. พล็อตข้อมูลจุดความร้อน
hotspot.plot(
    ax=ax,
    color="#d32f2f",
    markersize=1,
    alpha=0.5,
    label="GISTDA Hotspot Detected"
)

# 3. ตั้งค่าองค์ประกอบแผนที่อ้างอิงพิกัดองศา (WGS84)
ax.set_title("Hotspot Distribution", fontsize=14, pad=15)
ax.set_xlabel("Longitude (Degrees)")
ax.set_ylabel("Latitude (Degrees)")
ax.set_aspect("equal")

plt.grid(True, linestyle=":", alpha=0.3)
plt.legend(loc="lower left")
plt.show()